# Train st2 (Event Extraction) — BERT-large-NER

Third transformer in the multistep CausalSense pipeline.

**Task:** BIO token tagging — for each token in a sentence, predict one of `O` / `B-SUBJ` / `I-SUBJ` / `B-OBJ` / `I-OBJ`. Used to extract the subject and object event spans.

**Why this exists:** st0+st1 give you a sentence-level relation label. For an article-level causal graph you need the actual event spans (so coref can merge mentions across sentences). st2 produces those.

**Data:** same three files as st1.
- train: `data/Combined_dataset_CommonSense+News_Data/combined.csv`
- dev:   `data/News_data/dev.csv`
- test:  `data/Test_dataset/test.csv`

**Output:** a HuggingFace-format checkpoint folder (with BIO `id2label` baked into `config.json`).

**Runtime:** ~2–3 hr on a T4. Set Runtime → Change runtime type → T4 GPU.

## 0. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected — switch runtime to GPU.'
print('Device:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.44.2 scikit-learn pandas tqdm

## 2. Clone the repo

Public repo — no auth needed.

In [ ]:
import os

if not os.path.isdir('Relation_extraction'):
    !git clone https://github.com/eitang5/Relation_extraction.git
else:
    print('Repo already cloned.')

%cd Relation_extraction

## 3. Mount Google Drive (recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/causalsense/checkpoints

## 4. Sanity run — small subset, 1 epoch on bert-base-NER (~2–3 min)

Trains on 200/50/50 rows for one epoch to confirm the pipeline works end-to-end.

Expected output: BIO classification report at the end. Most labels will hit 0 with 200 rows × 1 epoch (the `O` class dominates — most tokens aren't part of any span). Goal: "no crash," not "good numbers."

In [ ]:
import os
os.environ['MODEL_NAME']  = 'dslim/bert-base-NER'
os.environ['OUTPUT_DIR']  = 'checkpoints/st2_sanity'
os.environ['EPOCHS']      = '1'
os.environ['BATCH_SIZE']  = '16'
os.environ['MAX_TRAIN']   = '200'
os.environ['MAX_DEV']     = '50'
os.environ['MAX_TEST']    = '50'

!python separate_tasks/EE.py

## 5. Real run — bert-large-NER, 10 epochs, full data (~2–3 hr on T4)

Output goes to Drive. The script saves the best (highest dev macro-F1) checkpoint.

**If OOM:** drop `BATCH_SIZE` to 8 or 4. bert-large-NER is ~340M params — should fit at batch 16 on T4, but tight.

**Note:** clearing the `MAX_*` env vars from sanity is important — otherwise the real run caps the data too.

In [ ]:
import os
for k in ('MAX_TRAIN', 'MAX_DEV', 'MAX_TEST'):
    os.environ.pop(k, None)

os.environ['MODEL_NAME']  = 'dslim/bert-large-NER'
os.environ['OUTPUT_DIR']  = '/content/drive/MyDrive/causalsense/checkpoints/st2_bert_large_ner'
os.environ['EPOCHS']      = '10'
os.environ['BATCH_SIZE']  = '16'

!python separate_tasks/EE.py

## 6. Verify the checkpoint reloads

`id2label` should show the 5 BIO labels (`O`, `B-SUBJ`, `I-SUBJ`, `B-OBJ`, `I-OBJ`).

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

ckpt = os.environ['OUTPUT_DIR']
model = AutoModelForTokenClassification.from_pretrained(ckpt)
tokenizer = AutoTokenizer.from_pretrained(ckpt)

print('Loaded:', ckpt)
print('Num labels:', model.config.num_labels)
print('id2label  :', model.config.id2label)
print('Params (M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

## 7. Smoke-test inference

Runs the model on a few example sentences and prints token-by-token BIO predictions plus the recovered subject/object spans.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)
id2label = model.config.id2label

def extract_spans(text):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        preds = model(**enc).logits.argmax(-1)[0].cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    labels = [id2label[p] for p in preds]

    # Recover subject and object spans by walking the BIO tags
    spans = {'SUBJ': [], 'OBJ': []}
    cur_type, cur_toks = None, []
    for tok, lab in zip(tokens, labels):
        if tok in (tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token):
            continue
        if lab.startswith('B-'):
            if cur_type and cur_toks:
                spans[cur_type].append(tokenizer.convert_tokens_to_string(cur_toks).strip())
            cur_type = lab[2:]
            cur_toks = [tok]
        elif lab.startswith('I-') and cur_type == lab[2:]:
            cur_toks.append(tok)
        else:
            if cur_type and cur_toks:
                spans[cur_type].append(tokenizer.convert_tokens_to_string(cur_toks).strip())
            cur_type, cur_toks = None, []
    if cur_type and cur_toks:
        spans[cur_type].append(tokenizer.convert_tokens_to_string(cur_toks).strip())
    return list(zip(tokens, labels)), spans

samples = [
    'The earthquake caused widespread destruction in coastal areas.',
    'Heavy rain prevented the football match from taking place.',
    'She studied hard in order to pass the exam.',
    'Internet access enables employees to work remotely.',
]

for s in samples:
    tagged, spans = extract_spans(s)
    print(f'\n>>> {s}')
    print('  SUBJ:', spans['SUBJ'])
    print('  OBJ :', spans['OBJ'])
    # Uncomment to see token-by-token tags:
    # print('  tags:', [t for t in tagged if t[1] != 'O'])

## Next steps

With st0 + st1 + st2 trained, you have all three transformers needed for the **chunk-and-merge** article-level pipeline:

1. **Per sentence**: st0 filters → st1 classifies type → st2 extracts (subject, object) spans → triple `(subject, relation, object)`.
2. **Per article**: aggregate the triples from all sentences → run event coref to merge mentions of the same event ("the earthquake" / "the quake") → produces an article-level causal graph.
3. **Across articles**: link nodes across the corpus via entity resolution → corpus-level graph.

**To continue training this checkpoint on your own data later:**

```python
os.environ['MODEL_NAME'] = '/content/drive/MyDrive/causalsense/checkpoints/st2_bert_large_ner'
os.environ['TRAIN_FILE'] = '/content/drive/MyDrive/my_data/train.csv'  # cols: text, subject, object, relation
os.environ['DEV_FILE']   = '/content/drive/MyDrive/my_data/dev.csv'
os.environ['TEST_FILE']  = '/content/drive/MyDrive/my_data/test.csv'
os.environ['OUTPUT_DIR'] = '/content/drive/MyDrive/causalsense/checkpoints/st2_bert_large_ner_mydata'
os.environ['EPOCHS']     = '3'
!python separate_tasks/EE.py
```

**Caveat to remember:** st2's training labels are auto-created by exact-token-match between the `subject`/`object` strings and the tokenized sentence (`EE.py` line ~39). If your data has paraphrased subjects/objects that don't appear verbatim, those rows train with all-`O` labels. Worth grepping for low-coverage rows before training.